# R.O.A.D. Barbados — Kraken PP-OCRv6 pipeline

A **Kraken-first** competition notebook. The Qwen3-VL-8B LoRA run
(`ROAD_Barbados_HTR_SOTA_Runpod.ipynb`) landed at **0.90491 OOF / 0.90485 public**
after one fold + GRPO. First place sits at **0.9399**. GRPO added `+0.00041`.
Post-processing added `+0.00258`. The remaining gap is not a ranking trick —
the recogniser is still reading ~15 % of words wrong (OOF WER 0.150, CER 0.045).

This notebook switches architecture.

## Why PP-OCRv6 instead of another VLM

On the BHL historical OCR board (2 165 pages of 18th–19th-century books,
2026-09-03) **kraken PP-OCRv6 medium (15.9 M params, Apache-2.0)** reads at
CER 0.034 vs 0.051 for Qwen3.5-9B, with **0 % looped pages**. It is a CTC line
model: one label per image column, so it cannot invent a second copy of a
legal formula the way an autoregressive decoder can. The challenge labels are
diplomatic transcriptions of secretary hand, which is exactly the setting
where "literal CTC" beats "fluent VLM".

The starter-kit Kraken script is **not** this. It page-segments already-cropped
lines, binarises them, and continues a 2024 CATMuS model for one epoch. This
notebook:

1. Treats every crop as **one line** (no page segmentation).
2. Fine-tunes `small-models-for-glam/kraken-ppocrv6-medium` with `--resize new`
   so the 44-language softmax collapses onto the challenge charset.
3. Trains **all five group-aware folds** (16 M params, minutes per fold).
4. Trains the **small** sibling as a second CTC family (cheap diversity).
5. Decodes with photometric TTA, then ROVER + char-LM + lexicon, all tuned OOF.
6. Optionally **ensembles the Qwen submission** when it is on disk — the two
   families fail differently, which is what a confusion-network vote converts
   into points.
7. Emails you on start, each fold, and finish (and the traceback on a crash),
   using the same `.env` SMTP settings as the Qwen notebook.

## BHL board vs this challenge — do not pick one row

The BHL table is **zero-shot printed books**. This challenge is **fine-tuned
secretary-hand line crops**. Rankings do not transfer:

| Model | Why it is (or is not) the move here |
|---|---|
| **Qwen3-VL-8B, fine-tuned** | Already 0.905 after **one** fold. The unused lever is the other four folds, not a different VLM. |
| **kraken PP-OCRv6** | Best *second* member: CTC cannot loop; Apache-2.0; 16 M params so 5 folds are cheap. |
| dots.ocr / dots.mocr | Top of BHL, but the weights sit under a **custom** licence agreement, not a clean Apache/MIT. The organisers DQ'd churro-3B for exactly that class of problem. Do not submit it. |
| PaddleOCR-VL-1.6 | Apache-2.0, but a page/document VLM. This dataset is already line-cropped. |
| OvisOCR2 | Strong specialist; licence is not on the commercial allow-list we can defend. |

The submission that can actually reach ~0.94 is **5-fold Qwen + 5-fold Kraken,
ROVER-fused**. Run this notebook and the Qwen notebook with `competition_mode`
(all folds). Submit both singles and the ensemble; keep the public-board winner.

**Do not run this inside the Qwen kernel.** kraken 7.1 pins `torch>=2.9` and
will fight that environment. A fresh venv / pod template is the whole setup.

## Metric

`0.5 * (1 - weighted_WER) + 0.5 * (1 - weighted_CER)`, length-power `0.5`,
copied from the VLM notebook and unit-tested. Empty predictions score 0.

## Rules

| Topic | Ruling | This notebook |
|---|---|---|
| Model licence | Must permit commercial use | `kraken-ppocrv6-{medium,small}` are Apache-2.0; a licence guard refuses anything else |
| StackMix | Permitted on Train.csv images only | Optional photometric extra views; no test images |
| External data | Not allowed | Fine-tune uses Train.csv only |
| Self-training on test | Unresolved | Disabled |

In [ ]:
import os as _os
import subprocess
import sys

RUN_INSTALL = _os.environ.get("ROAD_HEADLESS") != "1"

if RUN_INSTALL:
    def pip(*args: str) -> int:
        print("pip", *args)
        return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                              check=False).returncode

    pip("--upgrade", "pip")
    pip("huggingface_hub>=0.34.0", "opencv-python-headless>=4.9", "pillow>=10.2",
        "rapidfuzz>=3.9", "scikit-learn>=1.4", "pandas>=2.1", "tqdm", "pyyaml",
        "tensorboard>=2.16")
    # kraken 7.1.1 is the first release that trains the PP-OCRv6 architecture.
    rc = pip("kraken==7.1.1")
    if rc != 0:
        print("[env] kraken==7.1.1 failed; trying kraken>=7.1.0")
        pip("kraken>=7.1.0")

In [ ]:
import gc
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata
import warnings
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch

print(f"python           {sys.version.split()[0]}")
print(f"torch            {torch.__version__}")
print(f"cuda available   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"gpu              {GPU_NAME} ({GPU_VRAM_GB:.1f} GB)")
else:
    GPU_NAME, GPU_VRAM_GB = "cpu", 0.0
    print("gpu              none - ketos will train on CPU (slow, but 16M params fit)")

try:
    import kraken
    print(f"kraken           {getattr(kraken, '__version__', 'unknown')}")
except Exception as exc:
    print(f"!! kraken import failed ({exc}). Re-run the install cell in a fresh kernel.")

## 2. Config

The levers that actually change a score, in the order to touch them:

1. `train_folds = (0, 1, 2, 3, 4)` — already the default. Five CTC models are
   cheap and the ROVER vote is the most reliable point on this dataset.
2. `also_train_small = True` — a second architecture in the same family.
3. `resize = "new"` — collapse the 44-language softmax onto Train.csv. Switch to
   `"union"` only if zero-shot already reads the challenge charset well.
4. `vlm_test_csv` — path to the Qwen `submission.csv`. Blended at test time
   when present; OOF-tuned if `vlm_oof_csv` is also present.

In [ ]:
def load_dotenv(path: str | Path = ".env") -> list:
    path = Path(path)
    if not path.exists():
        return []
    loaded = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and val and not os.environ.get(key):
            os.environ[key] = val
            loaded.append(key)
    return loaded


_ENV_KEYS = load_dotenv()
print(f"[env] loaded {len(_ENV_KEYS)} settings from .env: {', '.join(_ENV_KEYS) or '(none)'}")


@dataclass
class Config:
    work_dir: str = "."
    data_dir: str = "data"
    image_subdir: str = "images"
    out_dir: str = "artifacts_kraken"
    train_csv: str = "Train.csv"
    test_csv: str = "Test.csv"
    sample_sub_csv: str = "SampleSubmission.csv"
    images_url: str = "https://storage.googleapis.com/road-handwriting/images.zip"

    model_id: str = "small-models-for-glam/kraken-ppocrv6-medium"
    model_file: str = "medium.safetensors"
    small_model_id: str = "small-models-for-glam/kraken-ppocrv6-small"
    small_model_file: str = "small.safetensors"
    also_train_small: bool = True
    kraken_version: str = "7.1.1"

    n_folds: int = 5
    train_folds: tuple = (0, 1, 2, 3, 4)
    fold_seed: int = 20260703
    seed: int = 42
    deterministic: bool = True

    # ketos. PP-OCRv6 rejects `--freeze-backbone` (vgsl-only) and crashes on
    # `-q early`: Lightning then sets max_epochs=-1, `estimated_stepping_batches`
    # is inf, and the default AdamW+Muon cosine schedule does int(inf)
    # (kraken#792). Use a fixed epoch budget. Leave optimizer empty so ketos
    # keeps the architecture default (AdamW+Muon).
    resize: str = "new"                 # new | union  (see markdown above)
    epochs: int = 50
    min_epochs: int = 15
    quit: str = "fixed"                 # fixed | early  (early is unsafe on ppocrv6)
    lr: float = 3e-4
    warmup: int = 200
    freeze_backbone: int = 0            # 0 = omit; ppocrv6 does not support it
    batch_size: int = 32
    schedule: str = "cosine"
    optimizer: str = ""                 # "" = architecture default (AdamW+Muon)
    augment: bool = True
    line_height: int = 128              # PP-OCRv6 medium native height
    max_line_width: int = 0             # 0 = derive so ketos drops nothing
    min_line_width: int = 64
    min_line_height: int = 48
    max_upscale: float = 3.0
    patch_grid: int = 1
    ink_crop_horizontal: bool = True
    ink_crop_vertical: bool = False
    deskew: bool = True
    deskew_max_deg: float = 3.0
    clahe: bool = True
    background_flatten: bool = True

    tta_modes: tuple = ("identity", "dark", "light", "sharp")
    infer_batch: int = 64
    dataloader_workers: int = 8

    lm_order: int = 7
    lm_tune_iters: int = 400

    # Path to the Qwen submission (optional ensemble). Leave empty to skip.
    vlm_test_csv: str = "submission_road_htr.csv"
    vlm_oof_csv: str = ""               # ID,Target per OOF line, if you have it
    vlm_blend: float = 0.35             # used only when OOF VLM is missing

    hf_repo_id: str = ""
    notify_email: bool = True            # same .env SMTP keys as the Qwen notebook
    tensorboard: bool = True


CFG = Config()
if not CFG.hf_repo_id:
    CFG.hf_repo_id = os.environ.get("HF_REPO_ID", "").strip()
WORK = Path(CFG.work_dir).resolve()
OUT = WORK / CFG.out_dir
OUT.mkdir(parents=True, exist_ok=True)

PERMISSIVE = {
    "apache-2.0", "mit", "bsd", "bsd-2-clause", "bsd-3-clause",
    "cc0-1.0", "cc-by-4.0", "unlicense", "isc",
}
KNOWN_LICENCES = {
    "small-models-for-glam/kraken-ppocrv6-medium": ("apache-2.0", True),
    "small-models-for-glam/kraken-ppocrv6-small": ("apache-2.0", True),
    "small-models-for-glam/kraken-ppocrv6-tiny": ("apache-2.0", True),
}


def check_licence(model_id: str) -> str:
    licence, verdict = KNOWN_LICENCES.get(model_id, (None, None))
    source = "static table"
    try:
        from huggingface_hub import model_info
        card = (model_info(model_id).cardData or {})
        hub = str(card.get("license") or card.get("license_name") or "").strip().lower()
        if hub:
            licence, source, verdict = hub, "huggingface hub", hub in PERMISSIVE
    except Exception as exc:
        print(f"[licence] hub lookup failed for {model_id} ({type(exc).__name__})")
    if not licence:
        raise RuntimeError(f"unknown licence for {model_id}")
    print(f"[licence] {model_id}: {licence} -> {'OK' if verdict else 'BLOCKED'} ({source})")
    if not verdict:
        raise RuntimeError(f"{model_id} is '{licence}', which the organisers disallow")
    return licence


CFG_LICENCE = check_licence(CFG.model_id)
SMALL_LICENCE = check_licence(CFG.small_model_id) if CFG.also_train_small else None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)
print(f"[cfg] {CFG.model_id} | folds {CFG.train_folds} | resize {CFG.resize} "
      f"| {CFG.epochs} epochs {CFG.quit} | lr {CFG.lr:g}")

In [ ]:
from tqdm.auto import tqdm

INTERACTIVE = bool(getattr(sys.stdout, "isatty", lambda: False)())
PROGRESS_INTERVAL = float(os.environ.get("PROGRESS_INTERVAL", "30"))
RUN_T0 = time.time()


def fmt_hms(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m{s:02d}s" if h else (f"{m}m{s:02d}s" if m else f"{s}s")


def progress(iterable=None, *, total=None, desc="", unit="it", leave=True, **kw):
    opts = dict(desc=desc, unit=unit, leave=leave, total=total,
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} "
                           "[{elapsed}<{remaining}, {rate_fmt}]{postfix}")
    if INTERACTIVE:
        opts.update(ncols=100, mininterval=0.2)
    else:
        opts.update(ncols=80, ascii=True, mininterval=PROGRESS_INTERVAL, miniters=1)
    opts.update(kw)
    return tqdm(iterable, **opts)


STAGES: list[dict] = []


def stage(name: str) -> None:
    now = time.time()
    if STAGES:
        STAGES[-1]["seconds"] = now - STAGES[-1]["t0"]
    STAGES.append({"name": name, "t0": now, "seconds": None})
    print(f"\n{'=' * 78}\n[stage {len(STAGES)}] {name}    (+{fmt_hms(now - RUN_T0)})\n{'=' * 78}")


def stage_summary() -> pd.DataFrame:
    now = time.time()
    if STAGES and STAGES[-1]["seconds"] is None:
        STAGES[-1]["seconds"] = now - STAGES[-1]["t0"]
    total = max(now - RUN_T0, 1e-6)
    df = pd.DataFrame([{"stage": s["name"], "minutes": round((s["seconds"] or 0) / 60, 1),
                        "share": f"{100 * (s['seconds'] or 0) / total:5.1f}%"} for s in STAGES])
    print(f"\nwall clock {fmt_hms(total)}\n{df.to_string(index=False)}")
    return df

### 2.1 Email notifications

Same SMTP contract as the Qwen notebook: `SMTP_HOST`, `SMTP_PORT`, `SMTP_USER`,
`SMTP_PASSWORD`, `NOTIFY_EMAIL_TO` from `.env`. A mail failure never kills the run.

In [ ]:
import hashlib
import smtplib
import traceback
from email.message import EmailMessage

RUN_FINGERPRINT = {
    "run_id": hashlib.sha256(
        json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                    for k, v in asdict(CFG).items()}, sort_keys=True, default=str
                   ).encode()).hexdigest()[:12],
}


class Notifier:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.to = os.environ.get("NOTIFY_EMAIL_TO", "").strip()
        self.host = os.environ.get("SMTP_HOST", "").strip()
        self.port = int(os.environ.get("SMTP_PORT", "587") or 587)
        self.user = os.environ.get("SMTP_USER", "").strip()
        self.password = os.environ.get("SMTP_PASSWORD", "").strip()
        self.enabled = bool(cfg.notify_email and self.to and self.host
                            and self.user and self.password)
        self.t0 = time.time()
        if self.enabled:
            print(f"[notify] email -> {self.to} via {self.host}:{self.port}")
        elif not cfg.notify_email:
            print("[notify] disabled (CFG.notify_email = False)")
        else:
            missing = [n for n, v in (("SMTP_HOST", self.host), ("SMTP_USER", self.user),
                                      ("SMTP_PASSWORD", self.password),
                                      ("NOTIFY_EMAIL_TO", self.to)) if not v]
            print(f"[notify] disabled (missing {', '.join(missing)} - see .env.example)")

    def _elapsed(self) -> str:
        m, s = divmod(int(time.time() - self.t0), 60)
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"

    def send(self, subject: str, body: str, urgent: bool = False) -> bool:
        prefix = "[ROAD-KRAKEN]" + (" FAILURE" if urgent else "")
        line = f"{prefix} {subject}"
        print(f"[notify] {line}")
        if not self.enabled:
            return False
        footer = (f"\n\n--\nrun {RUN_FINGERPRINT['run_id']} | {GPU_NAME} | "
                  f"elapsed {self._elapsed()}\nmodel {self.cfg.model_id}")
        msg = EmailMessage()
        msg["Subject"] = line
        msg["From"] = self.user
        msg["To"] = self.to
        msg.set_content(body + footer)
        try:
            with smtplib.SMTP(self.host, self.port, timeout=30) as server:
                server.starttls()
                server.login(self.user, self.password)
                server.send_message(msg)
            return True
        except Exception as exc:
            print(f"[notify] send failed ({type(exc).__name__}: {exc})")
            return False

    def run_started(self) -> None:
        self.send(
            f"kraken run started on {GPU_NAME}",
            f"Fine-tuning {self.cfg.model_id} on folds {list(self.cfg.train_folds)}.\n"
            f"resize {self.cfg.resize} | epochs {self.cfg.epochs} {self.cfg.quit} | "
            f"lr {self.cfg.lr:g} | also_small {self.cfg.also_train_small}\n"
            f"run id {RUN_FINGERPRINT['run_id']}",
        )

    def fold_done(self, tag: str, score: float, cer: float, wer: float) -> None:
        self.send(
            f"fold {tag} OOF {score:.5f}",
            f"tag    {tag}\nscore  {score:.5f}\nCER    {cer:.4f}\nWER    {wer:.4f}",
        )

    def finished(self, oof: float, baseline: float, path: Path) -> None:
        self.send(
            f"kraken run finished - OOF {oof:.5f}",
            f"out-of-fold score  {oof:.5f}\n"
            f"raw CTC identity   {baseline:.5f}  ({oof - baseline:+.5f} from post-processing)\n"
            f"submission         {path}",
        )

    def failed(self, where: str, detail: str) -> None:
        self.send(where, f"The run stopped.\n\n{detail}"[:8000], urgent=True)

    def install_crash_handler(self) -> None:
        def hook(exc_type, exc, tb):
            self.failed("kraken run crashed",
                        "".join(traceback.format_exception(exc_type, exc, tb)))
            sys.__excepthook__(exc_type, exc, tb)
        sys.excepthook = hook


NOTIFY = Notifier(CFG)
NOTIFY.install_crash_handler()
NOTIFY.run_started()

## 3. Data, labels, metric, split

In [ ]:
DATA = WORK / CFG.data_dir
DATA.mkdir(parents=True, exist_ok=True)


def download(url: str, dest: Path, chunk: int = 1 << 20) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[data] {dest.name} already present ({dest.stat().st_size / 1e6:.0f} MB)")
        return dest
    import urllib.request
    print(f"[data] downloading {url}")
    tmp = dest.with_suffix(dest.suffix + ".part")
    with urllib.request.urlopen(url) as resp, open(tmp, "wb") as fh:
        total = int(resp.headers.get("Content-Length", 0)) or None
        bar = progress(total=total, desc="download", unit="B",
                       unit_scale=True, unit_divisor=1024, bar_format=None)
        while True:
            buf = resp.read(chunk)
            if not buf:
                break
            fh.write(buf)
            bar.update(len(buf))
        bar.close()
    tmp.rename(dest)
    return dest


def ensure_images() -> Path:
    zip_path = DATA / "images.zip"
    img_dir = DATA / CFG.image_subdir
    if not (img_dir.exists() and any(img_dir.rglob("*.jpg"))):
        download(CFG.images_url, zip_path)
        print("[data] extracting ...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA)
    for root in [img_dir, DATA]:
        if root.exists():
            jpgs = list(root.rglob("*.jpg"))
            if jpgs:
                return jpgs[0].parent
    raise FileNotFoundError("No .jpg files found after extraction")


stage("data: download and index")
IMAGE_DIR = ensure_images()
IMAGE_INDEX = {p.stem: p for p in IMAGE_DIR.rglob("*")
               if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}}
print(f"[data] image dir  {IMAGE_DIR} | indexed {len(IMAGE_INDEX)}")


def image_path(image_id: str) -> Path | None:
    return IMAGE_INDEX.get(str(image_id).strip())


train_df = pd.read_csv(WORK / CFG.train_csv)
test_df = pd.read_csv(WORK / CFG.test_csv)
sample_sub = pd.read_csv(WORK / CFG.sample_sub_csv)
train_df["ID"] = train_df["ID"].astype(str).str.strip()
test_df["ID"] = test_df["ID"].astype(str).str.strip()

UNICODE_FOLD = {
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u2032": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
    "\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-",
    "\u2014": "-", "\u2015": "-", "\u2212": "-",
    "\u00a0": " ", "\u2007": " ", "\u202f": " ", "\u2009": " ", "\u200b": "",
    "\u2026": "...", "\ufeff": "",
}
_FOLD_RE = re.compile("|".join(map(re.escape, UNICODE_FOLD)))


def normalize_text(s: Any) -> str:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = unicodedata.normalize("NFC", str(s))
    s = _FOLD_RE.sub(lambda m: UNICODE_FOLD[m.group()], s)
    s = s.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return " ".join(s.split()).strip()


train_df["Target"] = train_df["Target"].map(normalize_text)
train_df = train_df[train_df["Target"].str.len() > 0].reset_index(drop=True)
CHARSET = sorted(Counter("".join(train_df["Target"])).keys())
CHAR_FREQ = Counter("".join(train_df["Target"]))
ALLOWED = set(CHARSET)
print(f"[labels] {len(train_df)} rows | {len(CHARSET)} chars | {''.join(CHARSET)!r}")


def restrict_charset(s: str) -> str:
    return "".join(ch for ch in s if ch in ALLOWED)

In [ ]:
from rapidfuzz.distance import Levenshtein

WEIGHT_FACTOR = 0.5


def _tokens(s: str, level: str) -> list:
    return list(s) if level == "char" else s.split()


def error_rates(refs: Sequence[str], hyps: Sequence[str], level: str,
                weight_factor: float = WEIGHT_FACTOR) -> dict:
    num_p = den_p = num_m = den_m = 0.0
    per_sample = []
    for ref, hyp in zip(refs, hyps):
        r = _tokens(ref, level)
        h = _tokens(hyp if isinstance(hyp, str) else "", level)
        n = max(len(r), 1)
        d = Levenshtein.distance(r, h)
        rate = d / n
        per_sample.append(rate)
        w = n ** weight_factor
        num_p += w * rate
        den_p += w
        num_m += d
        den_m += n
    return {"power": num_p / max(den_p, 1e-9), "micro": num_m / max(den_m, 1e-9),
            "macro": float(np.mean(per_sample)) if per_sample else 1.0,
            "per_sample": np.asarray(per_sample)}


def competition_score(refs: Sequence[str], hyps: Sequence[str],
                      weight_factor: float = WEIGHT_FACTOR, verbose: bool = False) -> dict:
    cer = error_rates(refs, hyps, "char", weight_factor)
    wer = error_rates(refs, hyps, "word", weight_factor)
    out = {"score": 0.5 * (1 - wer["power"]) + 0.5 * (1 - cer["power"]),
           "cer": cer["power"], "wer": wer["power"], "n": len(refs)}
    if verbose:
        print(f"score {out['score']:.5f} | CER {out['cer']:.4f} | WER {out['wer']:.4f} | n={out['n']}")
    return out


def _test_metric() -> None:
    assert abs(competition_score(["the quick brown fox"], ["the quick brown fox"])["score"] - 1) < 1e-9
    assert abs(competition_score(["abc def"], [""])["score"]) < 1e-9
    one = competition_score(["abcd"], ["abcx"])
    assert abs(one["cer"] - 0.25) < 1e-9 and abs(one["wer"] - 1.0) < 1e-9
    print("[metric] unit tests passed")


_test_metric()

In [ ]:
from rapidfuzz import process as rf_process
from sklearn.model_selection import StratifiedGroupKFold


def build_text_groups(texts: Sequence[str], threshold: float = 0.90) -> np.ndarray:
    keys = [" ".join(re.sub(r"[^a-z0-9 ]", " ", t.lower()).split()) for t in texts]
    n = len(keys)
    parent = list(range(n))

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)

    by_key: dict[str, list[int]] = defaultdict(list)
    for i, k in enumerate(keys):
        by_key[k].append(i)
    for idxs in by_key.values():
        for j in idxs[1:]:
            union(idxs[0], j)
    sim = rf_process.cdist(keys, keys, scorer=Levenshtein.normalized_similarity,
                           score_cutoff=threshold, workers=-1, dtype=np.float32)
    rows, cols = np.nonzero(sim)
    for i, j in zip(rows, cols):
        if i < j:
            union(int(i), int(j))
    roots = np.array([find(i) for i in range(n)])
    _, groups = np.unique(roots, return_inverse=True)
    return groups


train_df["group"] = build_text_groups(train_df["Target"].tolist())
train_df["n_chars"] = train_df["Target"].str.len()
train_df["n_words"] = train_df["Target"].str.split().apply(len)
train_df["len_bin"] = pd.qcut(train_df["n_chars"], q=6, labels=False, duplicates="drop")
sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.fold_seed)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train_df, train_df["len_bin"], train_df["group"])):
    train_df.loc[train_df.index[val_idx], "fold"] = fold
leaks = train_df.groupby("group")["fold"].nunique()
assert (leaks == 1).all(), "groups leak across folds"
print(train_df.groupby("fold").agg(n=("ID", "size"), med_chars=("n_chars", "median")))
print("[split] no group leaks")
train_df.to_csv(OUT / "train_folds.csv", index=False)


def fold_frames(fold: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    return (train_df[train_df.fold != fold].reset_index(drop=True),
            train_df[train_df.fold == fold].reset_index(drop=True))

## 4. Image pipeline

Same cleaning as the VLM notebook (flatten, deskew, horizontal ink-crop, CLAHE)
so a later ensemble is voting on reading, not on preprocessing. Height is 128 px
— the native PP-OCRv6 medium input — and width is **not** capped until ketos
sees the rescaled size, because ketos silently drops anything over `--max-width`.

In [ ]:
import cv2
from PIL import Image

cv2.setNumThreads(0)
CACHE_DIR = OUT / "cache" / f"h{CFG.line_height}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def read_gray(path: str | Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def flatten_background(gray: np.ndarray) -> np.ndarray:
    k = max(11, (min(gray.shape[:2]) // 2) | 1)
    k = min(k, 51) | 1
    bg = cv2.medianBlur(gray, k)
    out = cv2.divide(gray, np.maximum(bg, 1), scale=255)
    return np.clip(out, 0, 255).astype(np.uint8)


def ink_mask(gray: np.ndarray) -> np.ndarray:
    g = cv2.GaussianBlur(gray, (0, 0), 1.0)
    m = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              cv2.THRESH_BINARY_INV, 41, 12)
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))


def estimate_skew(gray: np.ndarray, max_deg: float = 3.0, step: float = 0.5) -> float:
    h, w = gray.shape[:2]
    scale = 64.0 / max(h, 1)
    small = cv2.resize(gray, (max(int(w * scale), 32), 64), interpolation=cv2.INTER_AREA)
    mask = (ink_mask(small) > 0).astype(np.float32)
    if mask.sum() < 20:
        return 0.0
    best_angle, best_score = 0.0, -1.0
    centre = (mask.shape[1] / 2, mask.shape[0] / 2)
    for angle in np.arange(-max_deg, max_deg + 1e-6, step):
        if abs(angle) < 1e-6:
            rot = mask
        else:
            M = cv2.getRotationMatrix2D(centre, float(angle), 1.0)
            rot = cv2.warpAffine(mask, M, (mask.shape[1], mask.shape[0]),
                                 flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(rot.sum(axis=1)))
        if score > best_score:
            best_score, best_angle = score, float(angle)
    return best_angle


def rotate_keep(gray: np.ndarray, angle: float) -> np.ndarray:
    if abs(angle) < 1e-3:
        return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)


def horizontal_ink_crop(gray: np.ndarray, pad_frac: float = 0.02) -> np.ndarray:
    m = ink_mask(gray)
    col = m.sum(axis=0)
    thr = max(col.max() * 0.02, 1.0)
    on = np.nonzero(col > thr)[0]
    if on.size < 2:
        return gray
    pad = int(gray.shape[0] * pad_frac) + 2
    a = max(int(on[0]) - pad, 0)
    b = min(int(on[-1]) + pad, gray.shape[1])
    if b - a < gray.shape[0]:
        return gray
    return gray[:, a:b]


def target_size(w: int, h: int, cfg: Config, scale: float = 1.0) -> tuple[int, int]:
    s = min((cfg.line_height * scale) / max(h, 1), cfg.max_upscale)
    tw, th = w * s, h * s
    cap = cfg.max_line_width or 10 ** 6
    if tw > cap:
        k = cap / tw
        tw, th = tw * k, th * k
    th = max(int(round(th)), cfg.min_line_height)
    tw = max(int(round(tw)), cfg.min_line_width)
    return int(tw), int(th)


def normalize_line(gray: np.ndarray, cfg: Config, scale: float = 1.0) -> np.ndarray:
    if cfg.background_flatten:
        gray = flatten_background(gray)
    if cfg.deskew:
        gray = rotate_keep(gray, estimate_skew(gray, cfg.deskew_max_deg))
    if cfg.ink_crop_horizontal:
        gray = horizontal_ink_crop(gray)
    if cfg.clahe:
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    h, w = gray.shape[:2]
    tw, th = target_size(w, h, cfg, scale)
    interp = cv2.INTER_AREA if tw < w else cv2.INTER_CUBIC
    return cv2.resize(gray, (tw, th), interpolation=interp)


def photometric_variant(img: np.ndarray, mode: str) -> np.ndarray:
    if mode == "identity":
        return img
    if mode == "dark":
        f = np.clip(255.0 * np.power(img / 255.0, 1.35), 0, 255).astype(np.uint8)
        return cv2.erode(f, np.ones((2, 2), np.uint8))
    if mode == "light":
        f = np.clip(255.0 * np.power(img / 255.0, 0.78), 0, 255).astype(np.uint8)
        return cv2.dilate(f, np.ones((2, 2), np.uint8))
    if mode == "sharp":
        blur = cv2.GaussianBlur(img, (0, 0), 1.2)
        return np.clip(cv2.addWeighted(img, 1.6, blur, -0.6, 0), 0, 255).astype(np.uint8)
    raise ValueError(mode)


def load_line(image_id: str, cfg: Config = None, scale: float = 1.0,
              use_cache: bool = True) -> np.ndarray:
    cfg = cfg or CFG
    if use_cache and abs(scale - 1.0) < 1e-6:
        cached = CACHE_DIR / f"{image_id}.png"
        if cached.exists():
            img = cv2.imread(str(cached), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                return img
    path = image_path(image_id)
    if path is None:
        return np.full((cfg.line_height, cfg.min_line_width), 235, np.uint8)
    out = normalize_line(read_gray(path), cfg, scale)
    if use_cache and abs(scale - 1.0) < 1e-6:
        cv2.imwrite(str(CACHE_DIR / f"{image_id}.png"), out)
    return out


from concurrent.futures import ThreadPoolExecutor

stage("image cache")
todo = [i for i in list(train_df.ID) + list(test_df.ID)
        if not (CACHE_DIR / f"{i}.png").exists()]
if todo:
    with ThreadPoolExecutor(12) as ex:
        list(progress(ex.map(lambda i: load_line(i), todo), total=len(todo),
                      desc="cache", unit="img"))
print(f"[cache] {CACHE_DIR} ready")

## 5. Fine-tune PP-OCRv6

ketos `path` format: `<id>.png` next to `<id>.gt.txt`. The whole crop is one
bbox line — the starter kit's page segmenter is the wrong tool here.

`--resize new` rebuilds the CTC head on the challenge alphabet. The pretrained
44-language softmax would spend most of its mass on scripts that never appear
in Barbados secretary hand, and (ketos docs) "rapidly unlearn missing labels"
if left in place. Do **not** pass `--freeze-backbone`: that flag is vgsl-only
and ketos rejects it for `ppocrv6`. Do **not** use `-q early`: Lightning then
sets `max_epochs=-1`, `estimated_stepping_batches` is inf, and the Muon cosine
schedule does `int(inf)` (kraken#792). `-q fixed -N 50` is the working recipe.
`--schedule cosine` matches the original PP-OCRv6 training.

Every fold is trained. A 16 M model on 3 300 lines is minutes, not hours, and
five independent hypotheses are the single biggest lever the VLM run left on
the table (it trained fold 0 only).

In [ ]:
from huggingface_hub import hf_hub_download

KRAKEN_ROOT = OUT / "kraken"
LINES = KRAKEN_ROOT / "lines"
MODELS = KRAKEN_ROOT / "models"
LOGS = KRAKEN_ROOT / "logs"
PREDS = KRAKEN_ROOT / "preds"
for d in (KRAKEN_ROOT, LINES, MODELS, LOGS, PREDS, KRAKEN_ROOT / "jobs"):
    d.mkdir(parents=True, exist_ok=True)


def write_line_gt(ids: Sequence[str], texts: Sequence[str] | None,
                  dest: Path, mode: str = "identity") -> list[Path]:
    dest.mkdir(parents=True, exist_ok=True)
    texts = list(texts) if texts is not None else [None] * len(ids)
    paths = []
    for image_id, text in zip(ids, texts):
        png = dest / f"{image_id}.png"
        if not png.exists():
            img = load_line(image_id)
            if mode != "identity":
                img = photometric_variant(img, mode)
            cv2.imwrite(str(png), img)
        if text is not None:
            # ketos trains on NFD; we score NFC. Both are lossless for this charset.
            (dest / f"{image_id}.gt.txt").write_text(
                unicodedata.normalize("NFD", text), encoding="utf-8")
        paths.append(png)
    return paths


def write_manifest(paths: Sequence[Path], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(str(Path(p).resolve()) for p in paths) + "\n", encoding="utf-8")
    return path


# NRTR positional encoding is 2048 tokens; PP-OCRv6 width stride is 8.
# ketos raises if max_width // 8 + 2 exceeds that.
PPOCRV6_NRTR_MAX_WIDTH = (2048 - 2) * 8  # 16368
PPOCRV6_DEFAULT_MAX_WIDTH = 2560
UNSUPPORTED_FLAG_RE = re.compile(r"Option (--[\w-]+) is not supported")


def width_report(paths: Sequence[Path], height: int) -> dict:
    widths = []
    for p in paths:
        with Image.open(p) as im:
            w, h = im.size
        if w < 1 or h < 1:
            continue
        widths.append(w * height / h)
    widths = np.asarray(widths, dtype=float) if len(widths) else np.array([PPOCRV6_DEFAULT_MAX_WIDTH], dtype=float)
    finite = widths[np.isfinite(widths)]
    if finite.size == 0:
        finite = np.array([PPOCRV6_DEFAULT_MAX_WIDTH], dtype=float)
    auto = int(math.ceil((float(finite.max()) + 64) / 64.0) * 64)
    limit = CFG.max_line_width or auto
    if not math.isfinite(limit) or limit < 1:
        limit = PPOCRV6_DEFAULT_MAX_WIDTH
    limit = int(min(limit, PPOCRV6_NRTR_MAX_WIDTH))
    return {"n": int(len(widths)), "max": int(finite.max()),
            "p99": float(np.percentile(finite, 99)), "max_width": limit,
            "over_limit": int((finite > limit).sum())}


def ketos_bin() -> str:
    found = shutil.which("ketos")
    if found:
        return found
    cand = Path(sys.executable).parent / ("ketos.exe" if os.name == "nt" else "ketos")
    if cand.exists():
        return str(cand)
    raise FileNotFoundError("ketos not on PATH - re-run the install cell")


def ketos_train_cmd(base: Path, train_m: Path, val_m: Path, out_dir: Path,
                    max_width: int) -> list:
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    precision = "bf16-mixed" if (torch.cuda.is_available()
                                 and torch.cuda.is_bf16_supported()) else "32-true"
    # kraken#792: -q early => Lightning max_epochs=-1 => int(inf) in the
    # ppocrv6 Muon cosine schedule. Always train a fixed budget on this arch.
    quit = "fixed" if CFG.quit == "early" else (CFG.quit or "fixed")
    if quit != CFG.quit:
        print(f"[train] ppocrv6 cannot use -q {CFG.quit} (kraken#792); "
              f"using -q {quit} -N {CFG.epochs}")
    try:
        max_w = int(max_width)
    except (TypeError, OverflowError, ValueError):
        max_w = PPOCRV6_DEFAULT_MAX_WIDTH
    if not math.isfinite(max_w) or max_w < 1:
        max_w = PPOCRV6_DEFAULT_MAX_WIDTH
    max_w = min(max_w, PPOCRV6_NRTR_MAX_WIDTH)
    cmd = [ketos_bin(), "-v", "-d", device, "--precision", precision,
           "--workers", str(min(8, max(1, CFG.dataloader_workers))),
           "-s", str(CFG.seed),
           "train", "-f", "path", "-i", str(base), "--resize", CFG.resize,
           "-u", "NFD", "--normalize-whitespace",
           "-t", str(train_m), "-e", str(val_m), "-o", str(out_dir),
           "-B", str(CFG.batch_size), "-N", str(CFG.epochs),
           "-q", quit, "-F", "1",
           "--lrate", str(CFG.lr), "--warmup", str(CFG.warmup),
           "--max-width", str(max_w)]
    if CFG.schedule:
        cmd += ["--schedule", CFG.schedule]
    if CFG.optimizer:
        cmd += ["--optimizer", CFG.optimizer]
    # `--freeze-backbone` is vgsl-only; ketos refuses it for architecture ppocrv6.
    cmd.append("--augment" if CFG.augment else "--no-augment")
    return cmd


FLAG_WITH_VALUE = {"--min-epochs", "--schedule", "--optimizer", "--freeze-backbone"}
FLAG_BARE = {"--normalize-whitespace"}


def drop_ketos_flag(cmd: list, flag: str) -> list:
    """Remove one CLI flag (and its value, if it takes one)."""
    out, skip = [], False
    for tok in cmd:
        if skip:
            skip = False
            continue
        if tok == flag:
            skip = flag in FLAG_WITH_VALUE and flag not in FLAG_BARE
            continue
        out.append(tok)
    return out


def force_quit_fixed(cmd: list) -> list:
    out = list(cmd)
    if "-q" in out:
        out[out.index("-q") + 1] = "fixed"
    return drop_ketos_flag(out, "--min-epochs")


def best_weights(out_dir: Path) -> Path | None:
    found = []
    for p in Path(out_dir).glob("best_*.safetensors"):
        try:
            found.append((float(p.stem.split("_", 1)[1]), p))
        except ValueError:
            continue
    if found:
        return max(found)[1]
    safes = sorted(Path(out_dir).glob("*.safetensors"), key=lambda p: p.stat().st_mtime)
    return safes[-1] if safes else None


def run_logged(cmd: list, log_path: Path, echo_every: float = 30.0) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("[ketos] $", " ".join(map(str, cmd))[:500])
    last = 0.0
    with open(log_path, "a", encoding="utf-8") as fh:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, errors="replace")
        for line in proc.stdout:
            fh.write(line)
            now = time.time()
            if INTERACTIVE or now - last > echo_every or "error" in line.lower():
                print(f"  {line.rstrip()[:220]}")
                last = now
        return proc.wait()


WORKER = r'''
import json, math, sys
from pathlib import Path
from PIL import Image
from kraken.containers import BBoxLine, Segmentation
from kraken.configs import RecognitionInferenceConfig
from kraken.tasks import RecognitionTaskModel

job = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
model = RecognitionTaskModel.load_model(job["model"])
dev = job.get("device", "cpu")
if str(dev).startswith("cuda"):
    accelerator, devices = "gpu", [int(dev.split(":")[1]) if ":" in str(dev) else 0]
else:
    accelerator, devices = "cpu", 1
config = RecognitionInferenceConfig(batch_size=job["batch"], num_line_workers=0,
                                    accelerator=accelerator, device=devices)
items, gap, out = job["items"], 16, []
for start in range(0, len(items), job["batch"]):
    chunk = items[start:start + job["batch"]]
    ims = [Image.open(it["path"]).convert("L") for it in chunk]
    width = max(im.width for im in ims)
    height = sum(im.height for im in ims) + gap * (len(ims) + 1)
    canvas = Image.new("L", (width, height), 255)
    lines, y = [], gap
    for k, im in enumerate(ims):
        canvas.paste(im, (0, y))
        lines.append(BBoxLine(id=f"l{k}", bbox=(0, y, im.width, y + im.height)))
        y += im.height + gap
    seg = Segmentation(type="bbox", imagename="canvas", text_direction="horizontal-lr",
                       script_detection=False, lines=lines)
    recs = list(model.predict(im=canvas, segmentation=seg, config=config))
    for it, rec in zip(chunk, recs):
        conf = [max(float(c), 1e-6) for c in (getattr(rec, "confidences", None) or [])]
        score = sum(math.log(c) for c in conf) / len(conf) if conf else -5.0
        pred = rec.prediction if rec.prediction is not None else ""
        out.append({"ID": it["id"], "text": pred, "score": score})
    print(f"decode {min(start + len(chunk), len(items))}/{len(items)}", flush=True)
Path(job["out"]).write_text(json.dumps(out), encoding="utf-8")
'''


def decode_model(model: Path, ids: Sequence[str], tag: str) -> pd.DataFrame:
    """Identity + photometric TTA. Each view is one hypothesis with a CTC confidence."""
    frames = []
    worker = KRAKEN_ROOT / "worker.py"
    worker.write_text(WORKER, encoding="utf-8")
    for mode in CFG.tta_modes:
        dest = LINES if mode == "identity" else KRAKEN_ROOT / f"lines_{mode}"
        paths = write_line_gt(ids, None, dest, mode=mode)
        job = KRAKEN_ROOT / "jobs" / f"{tag}_{mode}.json"
        res = KRAKEN_ROOT / "jobs" / f"{tag}_{mode}.out.json"
        job.write_text(json.dumps({
            "model": str(model), "batch": int(CFG.infer_batch), "out": str(res),
            "device": "cuda:0" if torch.cuda.is_available() else "cpu",
            "items": [{"id": i, "path": str(p.resolve())} for i, p in zip(ids, paths)],
        }), encoding="utf-8")
        rc = run_logged([sys.executable, str(worker), str(job)], LOGS / f"decode_{tag}.log")
        if rc != 0 or not res.exists():
            raise RuntimeError(f"decode {tag}/{mode} failed rc={rc}")
        df = pd.DataFrame(json.loads(res.read_text(encoding="utf-8")))
        df["view"] = f"{tag}:{mode}"
        df["text"] = df["text"].fillna("").map(
            lambda s: normalize_text(unicodedata.normalize("NFC", s)))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def top1_score(df: pd.DataFrame, ids: Sequence[str]) -> dict:
    ident = (df[df.view.str.endswith(":identity")]
             if "view" in df.columns else df).drop_duplicates("ID").set_index("ID")["text"]
    refs = train_df.set_index("ID").loc[list(ids), "Target"].tolist()
    return competition_score(refs, [ident.get(i, "") for i in ids])


def download_base(model_id: str, filename: str) -> Path:
    path = Path(hf_hub_download(model_id, filename, local_dir=str(KRAKEN_ROOT / "base")))
    print(f"[model] {model_id} -> {path}")
    return path


stage("prepare line ground truth")
all_paths = write_line_gt(train_df.ID.tolist(), train_df.Target.tolist(), LINES)
widths = width_report(all_paths, CFG.line_height)
print(f"[data] {widths['n']} lines at h={CFG.line_height}: max {widths['max']}px "
      f"p99 {widths['p99']:.0f} -> --max-width {widths['max_width']}")
if widths["over_limit"]:
    print(f"!! {widths['over_limit']} lines exceed max-width and will be DROPPED")

BASE = download_base(CFG.model_id, CFG.model_file)
SMALL_BASE = download_base(CFG.small_model_id, CFG.small_model_file) if CFG.also_train_small else None

In [ ]:
REPORT: dict = {}
DECODES: list[dict] = []


def train_one(tag: str, base: Path, tr_df: pd.DataFrame, va_df: pd.DataFrame) -> Path:
    out_dir = MODELS / tag
    weights = best_weights(out_dir)
    if weights is not None:
        print(f"[train] {tag}: reusing {weights.name}")
        return weights
    t0 = time.time()
    tr_m = write_manifest([LINES / f"{i}.png" for i in tr_df.ID],
                          KRAKEN_ROOT / "manifests" / f"{tag}_train.txt")
    va_m = write_manifest([LINES / f"{i}.png" for i in va_df.ID],
                          KRAKEN_ROOT / "manifests" / f"{tag}_val.txt")
    out_dir.mkdir(parents=True, exist_ok=True)
    (LOGS / f"train_{tag}.log").unlink(missing_ok=True)
    cmd = ketos_train_cmd(base, tr_m, va_m, out_dir, widths["max_width"])
    rc = run_logged(cmd, LOGS / f"train_{tag}.log")
    log_txt = (LOGS / f"train_{tag}.log").read_text(encoding="utf-8", errors="replace")
    if rc != 0:
        # Drop only the flag ketos named as unsupported (e.g. --freeze-backbone
        # on ppocrv6). Do not strip --optimizer/--schedule: that silently
        # switches to AdamW+Muon and then -q early hits kraken#792 (int(inf)).
        hit = UNSUPPORTED_FLAG_RE.search(log_txt)
        if hit:
            cmd = drop_ketos_flag(cmd, hit.group(1))
            print(f"[train] {tag}: retry without {hit.group(1)}")
            rc = run_logged(cmd, LOGS / f"train_{tag}.log")
            log_txt = (LOGS / f"train_{tag}.log").read_text(encoding="utf-8", errors="replace")
    if rc != 0 and ("cannot convert float infinity to integer" in log_txt
                    or "OverflowError" in log_txt):
        cmd = force_quit_fixed(cmd)
        print(f"[train] {tag}: retry with -q fixed (kraken#792)")
        rc = run_logged(cmd, LOGS / f"train_{tag}.log")
    weights = best_weights(out_dir)
    log = (LOGS / f"train_{tag}.log").read_text(encoding="utf-8", errors="replace")
    dropped = log.count("Failed. Replacing")
    if dropped:
        print(f"!! {tag}: ketos replaced {dropped} unusable samples")
    if rc != 0 or weights is None:
        raise RuntimeError(f"ketos {tag} failed rc={rc}; see {LOGS / f'train_{tag}.log'}")
    print(f"[train] {tag}: {weights.name} in {fmt_hms(time.time() - t0)}")
    return weights


def run_family(name: str, base: Path, family: str) -> list[dict]:
    out = []
    stage(f"zero-shot {name}")
    zs_ids = train_df.ID.tolist()[: min(800, len(train_df))]
    zs_path = PREDS / f"zeroshot_{name}.csv"
    if zs_path.exists():
        zs = pd.read_csv(zs_path, keep_default_na=False)
    else:
        zs = decode_model(base, zs_ids, f"zs_{name}")
        zs.to_csv(zs_path, index=False)
    REPORT[f"zeroshot_{name}"] = top1_score(zs, zs_ids)
    print(f"[zero-shot] {name}: score {REPORT[f'zeroshot_{name}']['score']:.5f} "
          f"CER {REPORT[f'zeroshot_{name}']['cer']:.4f} WER {REPORT[f'zeroshot_{name}']['wer']:.4f}")
    zs = REPORT[f"zeroshot_{name}"]
    NOTIFY.send(f"zero-shot {name} {zs['score']:.5f}",
                f"{name} unscored-on-Barbados (zero-shot) "
                f"score {zs['score']:.5f} CER {zs['cer']:.4f} WER {zs['wer']:.4f}")

    stage(f"fine-tune {name} on {len(CFG.train_folds)} fold(s)")
    for fold in progress(CFG.train_folds, desc=name, unit="fold"):
        tag = f"{name}_fold{fold}"
        tr_df, va_df = fold_frames(fold)
        weights = train_one(tag, base, tr_df, va_df)
        oof_path, test_path = PREDS / f"oof_{tag}.csv", PREDS / f"test_{tag}.csv"
        if oof_path.exists() and test_path.exists():
            oof = pd.read_csv(oof_path, keep_default_na=False)
            test = pd.read_csv(test_path, keep_default_na=False)
        else:
            oof = decode_model(weights, va_df.ID.tolist(), f"{tag}_oof")
            test = decode_model(weights, test_df.ID.tolist(), f"{tag}_test")
            oof.to_csv(oof_path, index=False)
            test.to_csv(test_path, index=False)
        res = top1_score(oof, va_df.ID.tolist())
        REPORT[tag] = res
        print(f"[oof] {tag}: {res['score']:.5f}  CER {res['cer']:.4f}  WER {res['wer']:.4f}")
        NOTIFY.fold_done(tag, res["score"], res["cer"], res["wer"])
        out.append({"tag": tag, "family": family, "fold": fold,
                    "oof": oof, "test": test, "score": res["score"]})
    return out


DECODES = run_family("medium", BASE, "ctc_medium")
if SMALL_BASE is not None:
    DECODES += run_family("small", SMALL_BASE, "ctc_small")

## 6. Fusion

Each fold × TTA view is a hypothesis. Character ROVER repairs the single-glyph
disagreements that dominate CER; a fold-safe character LM and a frequency
lexicon catch the word errors that dominate WER (the VLM's real gap to first
place). If a Qwen `submission.csv` is on disk it becomes another ROVER member.
The mix is only OOF-tuned when `CFG.vlm_oof_csv` exists; otherwise a gated
blend (prefer Kraken unless the two strings are close) is used so a bad VLM
line cannot overwrite a good CTC one.

In [ ]:
class CharLM:
    def __init__(self, texts: Sequence[str], order: int = 7, alpha: float = 0.4):
        self.order, self.alpha = order, alpha
        self.ctx: dict[str, Counter] = defaultdict(Counter)
        self.ctx_total: Counter = Counter()
        self.uni: Counter = Counter()
        bos = "\x02"
        for t in texts:
            s = bos * (order - 1) + t + "\x03"
            for i in range(order - 1, len(s)):
                ch = s[i]
                self.uni[ch] += 1
                for k in range(order):
                    h = s[i - k:i]
                    self.ctx[h][ch] += 1
                    self.ctx_total[h] += 1
        self.n_uni = sum(self.uni.values())
        self.v_uni = len(self.uni) + 1

    def _logp(self, ctx: str, ch: str) -> float:
        penalty = 0.0
        for k in range(len(ctx), -1, -1):
            h = ctx[len(ctx) - k:] if k else ""
            c = self.ctx[h].get(ch, 0) if h in self.ctx else 0
            if c > 0:
                return math.log(c / self.ctx_total[h]) + penalty
            penalty += math.log(self.alpha)
        return math.log((self.uni.get(ch, 0) + 1) / (self.n_uni + self.v_uni)) + penalty

    def logprob(self, text: str) -> float:
        s = "\x02" * (self.order - 1) + text + "\x03"
        return sum(self._logp(s[i - self.order + 1:i], s[i])
                   for i in range(self.order - 1, len(s)))

    def logprob_per_char(self, text: str) -> float:
        return self.logprob(text) / max(len(text) + 1, 1)


class Lexicon:
    def __init__(self, texts: Sequence[str]):
        self.freq = Counter(w for t in texts for w in t.split())
        self.vocab = set(self.freq)
        self.lower: dict[str, list[str]] = defaultdict(list)
        for w in self.vocab:
            self.lower[w.lower()].append(w)
        self.choices = sorted(self.vocab)
        self.mean_chars = float(np.mean([len(t) for t in texts])) if len(texts) else 60.0
        self.std_chars = float(np.std([len(t) for t in texts])) or 1.0

    def oov_rate(self, text: str) -> float:
        words = text.split()
        return (sum(w not in self.vocab for w in words) / len(words)) if words else 1.0


def match_case(source: str, target: str) -> str:
    if source.isupper() and len(source) > 1:
        return target.upper()
    if source[:1].isupper():
        return target[:1].upper() + target[1:]
    return target


def repair_words(text: str, lex: Lexicon, min_sim: float = 88.0,
                 min_freq: int = 3, max_len_ratio: float = 0.34) -> str:
    out = []
    for word in text.split():
        core = word.strip(".,;:()\"'")
        if not core or core in lex.vocab or len(core) < 3:
            out.append(word)
            continue
        if lex.lower.get(core.lower()):
            out.append(word)
            continue
        budget = max(1, int(len(core) * max_len_ratio))
        cands = rf_process.extract(
            core, lex.choices, scorer=Levenshtein.normalized_similarity,
            processor=str.lower, score_cutoff=min_sim / 100.0, limit=8)
        viable = [(sim + 0.02 * math.log10(lex.freq[c] + 1), c)
                  for c, sim, _ in cands
                  if lex.freq[c] >= min_freq
                  and Levenshtein.distance(core.lower(), c.lower()) <= budget]
        if not viable:
            out.append(word)
            continue
        best = max(viable)[1]
        out.append(word.replace(core, match_case(core, best)))
    return " ".join(out)


def collapse_repeats(text: str, max_repeat: int = 3) -> str:
    words = text.split()
    for size in (1, 2, 3, 4):
        i = 0
        while i + size * (max_repeat + 1) <= len(words):
            block = words[i:i + size]
            reps, j = 1, i + size
            while j + size <= len(words) and words[j:j + size] == block:
                reps += 1
                j += size
            if reps > max_repeat:
                words = words[:i + size * max_repeat] + words[j:]
            i += size
    return " ".join(words)


def _align_to_pivot(pivot: list, hyp: list) -> tuple[list, list]:
    align: list = [""] * len(pivot)
    inserts: list = [[] for _ in range(len(pivot) + 1)]
    i = j = 0
    for op in Levenshtein.editops(pivot, hyp):
        while i < op.src_pos:
            align[i] = hyp[j]
            i += 1
            j += 1
        if op.tag == "replace":
            align[i] = hyp[op.dest_pos]
            i += 1
            j += 1
        elif op.tag == "delete":
            align[i] = ""
            i += 1
        elif op.tag == "insert":
            inserts[i].append(hyp[op.dest_pos])
            j += 1
    while i < len(pivot):
        align[i] = hyp[j] if j < len(hyp) else ""
        i += 1
        j += 1
    return align, inserts


def rover_fuse(hyps: Sequence[str], weights: Sequence[float], level: str = "char") -> str:
    pairs = [(h, w) for h, w in zip(hyps, weights) if isinstance(h, str)]
    if not pairs:
        return ""
    merged: dict[str, float] = defaultdict(float)
    for h, w in pairs:
        merged[h] += float(w)
    if len(merged) == 1:
        return next(iter(merged))
    items = sorted(merged.items(), key=lambda kv: -kv[1])
    total = sum(w for _, w in items)
    split = (lambda s: list(s)) if level == "char" else (lambda s: s.split())
    pivot = split(items[0][0])
    if not pivot:
        return items[0][0]
    slot_votes = [defaultdict(float) for _ in range(len(pivot))]
    gap_votes = [defaultdict(float) for _ in range(len(pivot) + 1)]
    for text, w in items:
        align, inserts = _align_to_pivot(pivot, split(text))
        for k, unit in enumerate(align):
            slot_votes[k][unit] += w
        for k, run in enumerate(inserts):
            gap_votes[k][tuple(run)] += w
    out: list = []
    for k in range(len(pivot) + 1):
        run, vote = max(gap_votes[k].items(), key=lambda kv: (kv[1], -len(kv[0])))
        if run and vote > total * 0.5:
            out.extend(run)
        if k < len(pivot):
            best_unit, _ = max(slot_votes[k].items(),
                               key=lambda kv: (kv[1], kv[0] == pivot[k], -len(kv[0])))
            if best_unit:
                out.append(best_unit)
    return "".join(out) if level == "char" else " ".join(out)


assert rover_fuse(["hello world", "hello world", "hallo world"], [1, 1, 1]) == "hello world"
print("[rover] ok")

In [ ]:
@dataclass
class PostConfig:
    w_score: float = 1.0
    w_votes: float = 0.45
    w_lm: float = 0.15
    w_oov: float = -0.30
    w_lenz: float = -0.02
    use_rover: bool = True
    rover_level: str = "char"
    rover_temp: float = 0.08
    repair: bool = True
    repair_sim: float = 90.0
    repair_freq: int = 3
    restrict_charset: bool = True
    collapse: bool = True
    use_vlm: bool = False
    w_vlm: float = 0.35


FOLD_LM: dict[int, CharLM] = {}
FOLD_LEX: dict[int, Lexicon] = {}
FOLD_OF = dict(zip(train_df.ID, train_df.fold))


def fold_resources(fold: int) -> tuple[CharLM, Lexicon]:
    if fold not in FOLD_LM:
        texts = train_df.loc[train_df.fold != fold, "Target"].tolist() if fold >= 0 \
            else train_df["Target"].tolist()
        FOLD_LM[fold] = CharLM(texts, order=CFG.lm_order)
        FOLD_LEX[fold] = Lexicon(texts)
    return FOLD_LM[fold], FOLD_LEX[fold]


def frame_to_nbest(df: pd.DataFrame, family: str) -> dict:
    out: dict[str, list] = defaultdict(list)
    for row in df.itertuples():
        out[row.ID].append({"text": normalize_text(row.text), "score": float(row.score),
                            "view": getattr(row, "view", family), "family": family})
    return dict(out)


def merge_nbest(base: dict, extra: dict) -> dict:
    merged = {k: list(v) for k, v in base.items()}
    for i, cands in extra.items():
        merged.setdefault(i, []).extend(cands)
    return merged


OOF_NBEST: dict = {}
TEST_NBEST: dict = {}
for d in DECODES:
    OOF_NBEST = merge_nbest(OOF_NBEST, frame_to_nbest(d["oof"], d["family"]))
    TEST_NBEST = merge_nbest(TEST_NBEST, frame_to_nbest(d["test"], d["family"]))

OOF_IDS = sorted({i for i in OOF_NBEST})
OOF_REFS = train_df.set_index("ID").loc[OOF_IDS, "Target"].tolist()


def load_vlm_csv(path: str, family: str = "vlm") -> dict:
    p = Path(path)
    if not p.exists():
        p = WORK / path
    if not p.exists():
        return {}
    df = pd.read_csv(p, keep_default_na=False)
    df["ID"] = df["ID"].astype(str).str.strip()
    df["Target"] = df["Target"].map(normalize_text)
    return {r.ID: [{"text": r.Target, "score": 0.0, "view": "vlm", "family": family}]
            for r in df.itertuples()}


VLM_TEST = load_vlm_csv(CFG.vlm_test_csv)
VLM_OOF = load_vlm_csv(CFG.vlm_oof_csv) if CFG.vlm_oof_csv else {}
if VLM_TEST:
    print(f"[vlm] loaded {len(VLM_TEST)} test lines from {CFG.vlm_test_csv}")
    TEST_NBEST = merge_nbest(TEST_NBEST, VLM_TEST)
if VLM_OOF:
    print(f"[vlm] loaded {len(VLM_OOF)} OOF lines from {CFG.vlm_oof_csv}")
    OOF_NBEST = merge_nbest(OOF_NBEST, VLM_OOF)
HAS_VLM_OOF = bool(VLM_OOF) and any(c.get("family") == "vlm" for i in OOF_IDS
                                    for c in OOF_NBEST.get(i, []))


def clean_candidate(text: str, restrict: bool, collapse: bool) -> str:
    text = normalize_text(text)
    if collapse:
        text = collapse_repeats(text)
    if restrict:
        text = normalize_text(restrict_charset(text))
    return text


def build_features(nbest: dict, ids: Sequence[str], fold_of: dict,
                   restrict: bool = True, collapse: bool = True) -> dict:
    feats = {}
    for image_id in progress(ids, desc="features", unit="line", leave=False):
        lm, lex = fold_resources(fold_of.get(image_id, -1))
        agg: dict[str, dict] = defaultdict(lambda: {"scores": [], "votes": 0.0, "vlm": 0.0})
        for c in nbest.get(image_id, []):
            text = clean_candidate(c["text"], restrict, collapse)
            rec = agg[text]
            rec["scores"].append(float(c["score"]))
            rec["votes"] += 1.0
            if c.get("family") == "vlm":
                rec["vlm"] += 1.0
        if not agg:
            agg[""] = {"scores": [-50.0], "votes": 1.0, "vlm": 0.0}
        texts, rows = [], []
        for text, rec in agg.items():
            s = np.asarray(rec["scores"], dtype=np.float64)
            rows.append([float(s.max()), float(s.mean()), math.log1p(rec["votes"]),
                         lm.logprob_per_char(text), lex.oov_rate(text),
                         abs((len(text) - lex.mean_chars) / lex.std_chars),
                         rec["vlm"]])
            texts.append(text)
        feats[image_id] = {"texts": texts, "M": np.asarray(rows, dtype=np.float64)}
    return feats


def combined_scores(M: np.ndarray, pc: PostConfig) -> np.ndarray:
    return (pc.w_score * (0.5 * M[:, 0] + 0.5 * M[:, 1])
            + pc.w_votes * M[:, 2]
            + pc.w_lm * M[:, 3]
            + pc.w_oov * M[:, 4]
            + pc.w_lenz * M[:, 5]
            + (pc.w_vlm * M[:, 6] if pc.use_vlm else 0.0))


def apply_post(feats: dict, ids: Sequence[str], pc: PostConfig,
               lexicon_for: Callable[[str], Lexicon] | None = None) -> dict:
    out = {}
    for image_id in ids:
        f = feats[image_id]
        texts, M = f["texts"], f["M"]
        s = combined_scores(M, pc)
        if pc.use_rover and len(texts) > 1:
            z = s - s.max()
            w = np.exp(z / max(pc.rover_temp, 1e-3))
            w = w / w.sum()
            keep = w > 0.01
            pred = (rover_fuse([t for t, k in zip(texts, keep) if k],
                               w[keep].tolist(), level=pc.rover_level)
                    if keep.sum() >= 2 else texts[int(np.argmax(s))])
        else:
            pred = texts[int(np.argmax(s))]
        if not pred.strip():
            for j in np.argsort(-s):
                if texts[j].strip():
                    pred = texts[j]
                    break
        if pc.repair and lexicon_for is not None:
            pred = repair_words(pred, lexicon_for(image_id),
                                min_sim=pc.repair_sim, min_freq=pc.repair_freq)
        out[image_id] = normalize_text(pred)
    return out


stage("post-processing weight search")
OOF_FEATS = build_features(OOF_NBEST, OOF_IDS, FOLD_OF)


def oof_score(pc: PostConfig) -> float:
    preds = apply_post(OOF_FEATS, OOF_IDS, pc,
                       lexicon_for=lambda i: fold_resources(FOLD_OF.get(i, -1))[1])
    return competition_score(OOF_REFS, [preds[i] for i in OOF_IDS])["score"]


base_pc = PostConfig(use_rover=False, repair=False, use_vlm=False)
best_pc, best_val = base_pc, oof_score(base_pc)
print(f"  identity-only / default weights  {best_val:.5f}")
rng = np.random.default_rng(0)
bar = progress(range(CFG.lm_tune_iters), desc="tune", unit="trial")
for it in bar:
    trial = PostConfig(
        w_score=float(rng.uniform(0.0, 2.0)),
        w_votes=float(rng.uniform(0.0, 1.5)),
        w_lm=float(rng.uniform(0.0, 1.2)),
        w_oov=float(rng.uniform(-1.5, 0.1)),
        w_lenz=float(rng.uniform(-0.3, 0.05)),
        use_rover=False, repair=False, use_vlm=HAS_VLM_OOF,
        w_vlm=float(rng.uniform(0.0, 1.0)) if HAS_VLM_OOF else 0.0,
    )
    val = oof_score(trial)
    if val > best_val:
        best_pc, best_val = trial, val
        bar.write(f"  it {it:4d}  {val:.5f}")
    bar.set_postfix_str(f"best {best_val:.5f}", refresh=False)
bar.close()

stage_b = [(best_pc, best_val, "rover off")]
for level in ("char", "word"):
    for temp in (0.03, 0.08, 0.2):
        trial = PostConfig(**{**asdict(best_pc), "use_rover": True,
                              "rover_level": level, "rover_temp": temp})
        val = oof_score(trial)
        stage_b.append((trial, val, f"{level} T={temp}"))
        print(f"  rover {level} T={temp} {val:.5f}")
best_pc, best_val, chosen = max(stage_b, key=lambda kv: kv[1])
print(f"[tune] rover {chosen} -> {best_val:.5f}")

stage_c = [(best_pc, best_val, "repair off")]
for sim in (86.0, 90.0, 94.0):
    for freq in (2, 5, 15):
        trial = PostConfig(**{**asdict(best_pc), "repair": True,
                              "repair_sim": sim, "repair_freq": freq})
        val = oof_score(trial)
        stage_c.append((trial, val, f"sim={sim} freq={freq}"))
best_pc, best_val, chosen = max(stage_c, key=lambda kv: kv[1])
print(f"[tune] repair {chosen} -> {best_val:.5f}")
POST = best_pc
print(json.dumps(asdict(POST), indent=2))

## 7. Submission

Two files are written:

* `submission_kraken.csv` — OOF-tuned Kraken (+ small) fusion. This is the one
  whose local score you can trust.
* `submission_ensemble.csv` — same, plus the Qwen file as an extra ROVER member
  when it exists. If we did not have Qwen OOF predictions, the mix is *gated*:
  a Qwen string only wins a line when it is within a small edit distance of the
  Kraken consensus. That stops a fluent hallucination from overwriting a
  literal CTC reading, which is the failure mode the BHL board documented.

In [ ]:
stage("submission")
TEST_IDS = test_df.ID.tolist()
fold_resources(-1)
TEST_FOLD_OF = {i: -1 for i in TEST_IDS}
TEST_FEATS = build_features(TEST_NBEST, TEST_IDS, TEST_FOLD_OF,
                            restrict=POST.restrict_charset, collapse=POST.collapse)
TEST_PRED = apply_post(TEST_FEATS, TEST_IDS, POST,
                       lexicon_for=lambda i: FOLD_LEX[-1])

# Gated VLM blend when we could not OOF-tune the mix.
if VLM_TEST and not HAS_VLM_OOF:
    n_replaced = 0
    for i in TEST_IDS:
        k = TEST_PRED[i]
        v = VLM_TEST.get(i, [{}])[0].get("text", "")
        if not v or v == k:
            continue
        # Close: let ROVER of the two strings pick the characters.
        sim = Levenshtein.normalized_similarity(k, v)
        if sim >= 0.80:
            fused = rover_fuse([k, v], [1.0 - CFG.vlm_blend, CFG.vlm_blend])
            if fused != k:
                TEST_PRED[i] = normalize_text(restrict_charset(fused))
                n_replaced += 1
        # Far apart: keep Kraken. A 0.90 VLM that invents "the said" is how you
        # lose CER against a specialist.
    print(f"[ensemble] gated VLM blend touched {n_replaced}/{len(TEST_IDS)} lines "
          f"(sim>=0.80); remainder stayed Kraken")

submission = pd.DataFrame({"ID": TEST_IDS, "Target": [TEST_PRED[i] for i in TEST_IDS]})
n_empty = int((submission.Target.str.strip() == "").sum())
if n_empty:
    print(f"!! {n_empty} empty predictions - filling from raw CTC top-1")
    fallback = {}
    for i, cands in TEST_NBEST.items():
        ctc = [c for c in cands if str(c.get("family", "")).startswith("ctc")]
        pool = ctc or cands
        fallback[i] = max(pool, key=lambda c: c["score"])["text"] if pool else "the"
    mask = submission.Target.str.strip() == ""
    submission.loc[mask, "Target"] = [normalize_text(fallback.get(i, "the")) or "the"
                                      for i in submission.loc[mask, "ID"]]

SUB_PATH = WORK / "submission_kraken.csv"
ENS_PATH = WORK / "submission_ensemble.csv"
submission.to_csv(SUB_PATH, index=False)
submission.to_csv(ENS_PATH, index=False)
print(f"wrote {SUB_PATH} ({len(submission)} rows)")

# Raw CTC top-1 OOF, for the ladder.
raw_preds = []
for image_id in OOF_IDS:
    cands = [c for c in OOF_NBEST[image_id] if str(c.get("family", "")).startswith("ctc")
             and str(c.get("view", "")).endswith(":identity")]
    raw_preds.append(cands[0]["text"] if cands else "")
RAW = competition_score(OOF_REFS, raw_preds)
_fused_map = apply_post(OOF_FEATS, OOF_IDS, POST,
                        lexicon_for=lambda j: fold_resources(FOLD_OF.get(j, -1))[1])
FUSED = competition_score(OOF_REFS, [_fused_map[i] for i in OOF_IDS])

print("=" * 78)
print("KRAKEN PP-OCRv6 RUN SUMMARY")
print("=" * 78)
print(f"backbone     {CFG.model_id}  ({CFG_LICENCE})")
print(f"resize       {CFG.resize} | folds {list(CFG.train_folds)} | also_small {CFG.also_train_small}")
for k, v in REPORT.items():
    if isinstance(v, dict) and "score" in v:
        print(f"  {k:<28} {v['score']:.5f}  CER {v['cer']:.4f}  WER {v['wer']:.4f}")
print(f"OOF raw CTC identity         {RAW['score']:.5f}")
print(f"OOF after post-processing    {FUSED['score']:.5f}  ({FUSED['score'] - RAW['score']:+.5f})")
print(f"Qwen VLM (this pod)          0.90491 OOF / 0.90485 public  (1 fold + GRPO)")
print(f"current first place          0.93989")
print(f"submission                   {SUB_PATH}")
if VLM_TEST:
    print(f"ensemble                     {ENS_PATH}  (gated VLM blend, "
          f"{'OOF-tuned' if HAS_VLM_OOF else 'not OOF-tuned'})")
print("=" * 78)
print(
    "If fused OOF is still below ~0.93 the remaining points are in the recogniser,\n"
    "not the fusion: train more epochs (`CFG.epochs = 80`), try `resize='union'`\n"
    "as a second family, or raise `line_height` to 160 with a larger max-width.\n"
    "Do not throw away the Qwen file - submit both and keep whichever the\n"
    "public leaderboard prefers; the private set can disagree."
)
stage_summary()
NOTIFY.finished(FUSED["score"], RAW["score"], SUB_PATH)
with open(OUT / "kraken_report.json", "w") as fh:
    json.dump({"report": {k: v for k, v in REPORT.items() if not isinstance(v, np.ndarray)},
               "raw": RAW, "fused": FUSED, "post": asdict(POST),
               "config": {k: (list(v) if isinstance(v, tuple) else v)
                          for k, v in asdict(CFG).items()}}, fh, indent=2, default=str)